# Generation evaluation

## 1. Setup

In [1]:
from pathlib import Path
import json
import os
import re
import sys
import time

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "doc_notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
REPO_ROOT = (
    PROJECT_ROOT.parent.parent
    if PROJECT_ROOT.name == "gatherly_rag" and PROJECT_ROOT.parent.name == "services"
    else PROJECT_ROOT
)
ROOT_ENV_FILE = REPO_ROOT / ".env"
if not ROOT_ENV_FILE.is_file():
    raise FileNotFoundError(f"Root .env not found: {ROOT_ENV_FILE}")
load_dotenv(ROOT_ENV_FILE, override=True)
if not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError(f"GEMINI_API_KEY is missing from {ROOT_ENV_FILE}")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import importlib

from doc_rag.rag_service import RagService

generation = importlib.import_module("doc_rag.11_generation")

QUERIES_FILE = (
    PROJECT_ROOT / "data" / "rag" / "evaluations" / "generation_evaluations_queries.json"
)
RESULTS_FILE = (
    PROJECT_ROOT / "data" / "rag" / "evaluations" / "final_generation_results.json"
)

# Set True once if you want to wipe the checkpoint and start over.
RESET_CACHE = False
JUDGE_MODEL = os.getenv("GENERATION_JUDGE_MODEL", "gemini-3.6-flash")

queries = json.loads(QUERIES_FILE.read_text(encoding="utf-8"))
print(f"Loaded {len(queries)} generation eval queries")
print(pd.DataFrame(queries)[["id", "expected_mode", "language"]].to_string(index=False))
print(f"Checkpoint / results file: {RESULTS_FILE}")
print(f"Environment loaded from: {ROOT_ENV_FILE}")
print("GEMINI_API_KEY loaded: yes")
print(f"RESET_CACHE={RESET_CACHE}")
print(f"Generator model: {generation.DEFAULT_LLM_MODEL}")
print(f"Independent judge model: {JUDGE_MODEL}")

Loaded 61 generation eval queries
         id expected_mode language
       SE01          text       en
       SE03          text       en
       SE04          text       en
       SE05          text       en
       SE10          text       en
      M2_01          text       fr
      M2_04          text       fr
       AW01          text       ar
       AW03          text       ar
       AW06          text       ar
       AW10          text       ar
    IMG_B01        visual       en
    IMG_B02        visual       en
    IMG_C01        visual       en
    IMG_C03        visual       en
    IMG_G02        visual       en
   IMG_RU01        visual       en
    IMG_M05        visual       en
   IMG_EF01        visual       en
   IMG_RW02        visual       en
   IMG_WW01        visual       en
       AW02          text       ar
       AW04          text       ar
       AW05          text       ar
       AW07          text       ar
       AW08          text       ar
       AW09          

In [2]:
service = RagService()
print("RagService ready")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RagService ready


## 2. Helpers

In [3]:
from openai import OpenAI
import os

def source_files(sources):
    return [str(s.get("file_name") or "") for s in sources]


def image_ids(images):
    return [str(i.get("image_id") or "") for i in images]


def acceptable_documents(item):
    return set(item.get("expected_documents") or [item.get("expected_document") or ""])


def relevant_pages_by_document(item):
    sources = item.get("relevant_sources") or []
    if sources:
        return {source["document"]: set(source.get("pages") or []) for source in sources}
    return {item.get("expected_document") or "": set(item.get("expected_pages") or [])}


def text_source_is_grounded(item, source):
    file_name = str(source.get("file_name") or "")
    if file_name not in acceptable_documents(item):
        return False
    pages = relevant_pages_by_document(item).get(file_name, set())
    if pages:
        page = source.get("page_number")
        return pd.notna(page) and int(page) in pages
    sections = {str(value).strip().casefold() for value in (item.get("expected_sections") or []) if str(value).strip()}
    if sections:
        section = str(source.get("section_title") or "").strip().casefold()
        return bool(section) and any(value in section or section in value for value in sections)
    keywords = [str(value).strip().casefold() for value in (item.get("answer_keywords") or []) if str(value).strip()]
    if keywords:
        text = str(source.get("text") or "").casefold()
        return any(value in text for value in keywords)
    return True


def refresh_cached_grounding(row):
    item = next((q for q in queries if q["id"] == row.get("id")), None)
    if item is None:
        return row
    text_sources = []
    pattern = re.compile(r"^\[T\d+\]\s+(.*?)\s+p\.(\d+):", re.MULTILINE)
    for file_name, page in pattern.findall(str(row.get("judge_context") or "")):
        text_sources.append({"file_name": file_name, "page_number": int(page)})
    images = [{"image_id": value} for value in (row.get("image_ids") or [])]
    all_files = set(row.get("text_files") or []) | set(row.get("image_files") or [])
    return row


def load_cached_rows() -> list[dict]:
    if RESET_CACHE and RESULTS_FILE.exists():
        RESULTS_FILE.unlink()
        print("Cache reset — deleted previous results file.")
        return []
    if not RESULTS_FILE.exists():
        return []
    try:
        payload = json.loads(RESULTS_FILE.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"Could not read cache ({exc}); starting empty.")
        return []
    rows = payload.get("results") if isinstance(payload, dict) else payload
    if not isinstance(rows, list):
        return []
    return [refresh_cached_grounding(r) for r in rows if isinstance(r, dict) and r.get("id")]


def row_is_complete(row: dict) -> bool:
    """Complete only when generation and all three judge scores succeeded."""
    required_scores = ("faithfulness", "correctness", "relevance")
    return (
        bool(str(row.get("answer") or "").strip())
        and all(row.get(name) is not None for name in required_scores)
    )


def save_checkpoint(rows: list[dict], summary: dict | None = None) -> None:
    # Keep query order from the eval set when possible.
    by_id = {r["id"]: r for r in rows}
    ordered = [by_id[q["id"]] for q in queries if q["id"] in by_id]
    payload = {
        "summary": summary or {"status": "in_progress", "n_cached": len(ordered)},
        "results": ordered,
    }
    RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    RESULTS_FILE.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def judge_answer(query: str, expected: str, answer: str, context: str) -> dict:
    """Anchored 1-4 judge for faithfulness, correctness, and relevance."""
    # client = generation.get_llm_client()
    client = OpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=os.getenv("GEMINI_API_KEY"),
    )

    prompt = (
        "You are a strict multilingual RAG evaluator. Return one JSON object only, "
        "with integer keys faithfulness, correctness, relevance and a short note.\n"
        "Use this anchored 1-4 rubric:\n"
        "Faithfulness: 1=unsupported or contradictory; 2=several unsupported claims; "
        "3=mostly supported with minor unsupported detail; 4=fully supported by context.\n"
        "Correctness: 1=mostly wrong; 2=major errors or omissions; "
        "3=mostly correct with minor issues; 4=matches the gold answer.\n"
        "Relevance: 1=does not answer; 2=partially relevant; "
        "3=answers the main request; 4=direct, complete and focused.\n"
        "Judge meaning across languages; do not require verbatim wording.\n\n"
        f"Question: {query}\n\n"
        f"Gold answer: {expected}\n\n"
        f"Retrieved context: {context}\n\n"
        f"Model answer: {answer}\n"
    )
    last_error = None
    for attempt in range(1, 4):
      try:
        resp = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=1000,
            response_format={"type": "json_object"},
        )
        raw = (resp.choices[0].message.content or "").strip()
        if not raw:
            raise ValueError("empty judge response")
        match = re.search(r"\{.*\}", raw, flags=re.S)
        data = json.loads(match.group(0) if match else raw)
        scores = {name: int(data[name]) for name in ("faithfulness", "correctness", "relevance")}
        if any(score not in {1, 2, 3, 4} for score in scores.values()):
            raise ValueError(f"scores outside 1-4: {scores}")
        return {**scores, "note": str(data.get("note", ""))[:300], "attempts": attempt}
      except Exception as exc:
        last_error = exc
        if attempt < 3:
            time.sleep(2 * attempt)
    return {
        "faithfulness": None, "correctness": None, "relevance": None,
        "note": f"judge_failed_after_3_attempts: {last_error}", "attempts": 3,
    }


def evaluate_one(item: dict) -> dict:
    q = item["query"]
    expected_mode = item["expected_mode"]
    expected_doc = item.get("expected_document") or ""
    expected_docs = set(item.get("expected_documents") or [expected_doc])

    t0 = time.perf_counter()
    result = service.ask(q)
    elapsed = time.perf_counter() - t0

    mode = result.get("mode") or ""
    texts = result.get("sources") or []
    images = result.get("image_sources") or []
    answer = result.get("answer") or ""

    text_files = source_files(texts)
    img_files = source_files(images)



    if expected_mode == "visual":
        visual_same_doc = (
            len(text_files) > 0 and all(f in expected_docs for f in text_files)
        )
        text_no_images = None
    else:
        visual_same_doc = None
        text_no_images = len(images) == 0

    context_parts = []
    for index, source in enumerate(texts[:5], start=1):
        source_text = str(source.get("text") or "").strip()
        if source_text:
            context_parts.append(
                f"[T{index}] {source.get('file_name')} p.{source.get('page_number')}: {source_text}"
            )
    judge_context = "\n\n".join(context_parts)[:12000]
    judge = judge_answer(q, item.get("expected_answer") or "", answer, judge_context)

    return {
        "id": item["id"],
        "query": q,
        "language": item.get("language"),
        "expected_mode": expected_mode,
        "predicted_mode": mode,
        "router_ok": mode == expected_mode,
        "expected_document": expected_doc,
        "visual_same_doc": visual_same_doc,
        "text_no_images": text_no_images,
        "faithfulness": judge["faithfulness"],
        "correctness": judge["correctness"],
        "relevance": judge["relevance"],
        "judge_note": judge["note"],
        "judge_attempts": judge["attempts"],
        "generator_model": generation.DEFAULT_LLM_MODEL,
        "judge_model": JUDGE_MODEL,
        "judge_context": judge_context,
        "variant_of": item.get("variant_of"),
        "cross_language": bool(item.get("cross_language", False)),
        "n_text_sources": len(texts),
        "n_image_sources": len(images),
        "text_files": text_files,
        "image_files": img_files,
        "image_ids": image_ids(images),
        "answer": answer,
        "seconds": round(elapsed, 2),
    }

## 3. Run evaluation

In [4]:
cached = load_cached_rows()
by_id = {r["id"]: r for r in cached if row_is_complete(r)}
rows = list(by_id.values())
print(f"Resuming with {len(by_id)}/{len(queries)} cached complete results")

stopped_early = False
for i, item in enumerate(queries, start=1):
    qid = item["id"]
    if qid in by_id:
        print(f"[{i}/{len(queries)}] {qid} — skip (cached)", flush=True)
        continue

    print(f"[{i}/{len(queries)}] {qid} ({item['expected_mode']}) ...", flush=True)
    try:
        row = evaluate_one(item)
    except Exception as exc:
        print(f"  STOPPED on error: {exc}", flush=True)
        print("  Progress saved. Re-run this cell later to continue.", flush=True)
        stopped_early = True
        break

    by_id[qid] = row
    rows = list(by_id.values())
    save_checkpoint(rows)
    print(
        f"  mode={row['predicted_mode']} router_ok={row['router_ok']} "
        f"{row['faithfulness']}/{row['correctness']}/{row['relevance']} "
        f"({row['seconds']}s) "
        f"[saved {len(by_id)}/{len(queries)}]",
        flush=True,
    )

# Prefer query-file order for the table.
rows = [by_id[q["id"]] for q in queries if q["id"] in by_id]
df = pd.DataFrame(rows)
if len(df):
    display(
        df[
            [
                "id",
                "expected_mode",
                "predicted_mode",
                "router_ok",
                "visual_same_doc",
                "text_no_images",
                "faithfulness",
                "correctness",
                "relevance",
                "seconds",
            ]
        ]
    )
else:
    print("No rows yet.")

if stopped_early:
    print(f"Partial run: {len(by_id)}/{len(queries)} done -> {RESULTS_FILE}")
elif len(by_id) < len(queries):
    missing = [q["id"] for q in queries if q["id"] not in by_id]
    print(f"Still missing {len(missing)}: {missing}")
else:
    print(f"All {len(queries)} queries complete.")

Resuming with 61/61 cached complete results
[1/61] SE01 — skip (cached)
[2/61] SE03 — skip (cached)
[3/61] SE04 — skip (cached)
[4/61] SE05 — skip (cached)
[5/61] SE10 — skip (cached)
[6/61] M2_01 — skip (cached)
[7/61] M2_04 — skip (cached)
[8/61] AW01 — skip (cached)
[9/61] AW03 — skip (cached)
[10/61] AW06 — skip (cached)
[11/61] AW10 — skip (cached)
[12/61] IMG_B01 — skip (cached)
[13/61] IMG_B02 — skip (cached)
[14/61] IMG_C01 — skip (cached)
[15/61] IMG_C03 — skip (cached)
[16/61] IMG_G02 — skip (cached)
[17/61] IMG_RU01 — skip (cached)
[18/61] IMG_M05 — skip (cached)
[19/61] IMG_EF01 — skip (cached)
[20/61] IMG_RW02 — skip (cached)
[21/61] IMG_WW01 — skip (cached)
[22/61] AW02 — skip (cached)
[23/61] AW04 — skip (cached)
[24/61] AW05 — skip (cached)
[25/61] AW07 — skip (cached)
[26/61] AW08 — skip (cached)
[27/61] AW09 — skip (cached)
[28/61] AW11 — skip (cached)
[29/61] AW12 — skip (cached)
[30/61] AW13 — skip (cached)
[31/61] AW14 — skip (cached)
[32/61] AW15 — skip (cached)
[

,id,expected_mode,predicted_mode,router_ok,visual_same_doc,text_no_images,faithfulness,correctness,relevance,seconds
0,SE01,text,text,True,None,True,4,2,4,13.61
1,SE03,text,text,True,None,True,4,4,4,8.18
2,SE04,text,text,True,None,True,4,4,4,13.01
3,SE05,text,visual,False,None,False,2,1,1,10.98
4,SE10,text,visual,False,None,False,3,2,3,13.59
...,...,...,...,...,...,...,...,...,...,...
56,XL_EN_AW03,text,text,True,None,True,4,3,4,12.19
57,XL_AR_M2_01,text,text,True,None,True,4,4,4,18.86
58,XL_AR_M2_04,text,text,True,None,True,4,4,4,13.80
59,XL_FR_SE01,text,text,True,None,True,4,2,4,12.33


All 61 queries complete.


In [5]:
# cached = load_cached_rows()

# print("Saved rows:", len(cached))

# for row in cached:
#     print("\nCASE:", row["id"])
#     print("Judge model:", row.get("judge_model"))
#     print("Judge error:", row.get("judge_note"))

## 4. Summary + save

In [6]:
# Rebuild from checkpoint so summary works even after a kernel restart.
rows = [r for r in load_cached_rows() if row_is_complete(r)]
by_id = {r["id"]: r for r in rows}
rows = [by_id[q["id"]] for q in queries if q["id"] in by_id]
df = pd.DataFrame(rows)

n = len(df)
visual = df[df["expected_mode"] == "visual"] if n else df
text = df[df["expected_mode"] == "text"] if n else df
cross_language = df[df["cross_language"]] if n else df
by_language = {}
if n:
    for language, group in df.groupby("language", dropna=False):
        by_language[str(language)] = {
            "n": int(len(group)),
            "faithfulness": float(group["faithfulness"].mean()),
            "correctness": float(group["correctness"].mean()),
            "relevance": float(group["relevance"].mean()),
        }

summary = {
    "generator_model": generation.DEFAULT_LLM_MODEL,
    "judge_model": JUDGE_MODEL,
    "n_queries": n,
    "n_expected": len(queries),
    "complete": n == len(queries),
    "router_accuracy": float(df["router_ok"].mean()) if n else 0.0,
    "mean_faithfulness": float(df["faithfulness"].mean()) if n else None,
    "mean_correctness": float(df["correctness"].mean()) if n else None,
    "mean_relevance": float(df["relevance"].mean()) if n else None,
    "text": {
        "n": int(len(text)),
        "router_accuracy": float(text["router_ok"].mean()) if len(text) else None,
        "text_no_images_rate": float(text["text_no_images"].mean()) if len(text) else None,
    },
    "visual": {
        "n": int(len(visual)),
        "router_accuracy": float(visual["router_ok"].mean()) if len(visual) else None,
        "visual_same_doc_rate": float(visual["visual_same_doc"].mean()) if len(visual) else None,
    },
    "cross_language": {
        "n": int(len(cross_language)),
        "router_accuracy": float(cross_language["router_ok"].mean()) if len(cross_language) else None,
        "faithfulness": float(cross_language["faithfulness"].mean()) if len(cross_language) else None,
        "correctness": float(cross_language["correctness"].mean()) if len(cross_language) else None,
        "relevance": float(cross_language["relevance"].mean()) if len(cross_language) else None,
    },
    "by_language": by_language,
}

save_checkpoint(rows, summary=summary)
print(json.dumps(summary, indent=2))
print(f"\nSaved -> {RESULTS_FILE}")
if n < len(queries):
    missing = [q["id"] for q in queries if q["id"] not in by_id]
    print(f"Incomplete: missing {len(missing)}: {missing}")
    print("Re-run the eval cell after the rate limit resets.")

{
  "generator_model": "gemini-3.5-flash",
  "judge_model": "gemini-3.6-flash",
  "n_queries": 61,
  "n_expected": 61,
  "complete": true,
  "router_accuracy": 0.9180327868852459,
  "mean_faithfulness": 3.7704918032786887,
  "mean_correctness": 3.5737704918032787,
  "mean_relevance": 3.901639344262295,
  "text": {
    "n": 51,
    "router_accuracy": 0.9607843137254902,
    "text_no_images_rate": 0.9607843137254902
  },
  "visual": {
    "n": 10,
    "router_accuracy": 0.7,
    "visual_same_doc_rate": 0.8
  },
  "cross_language": {
    "n": 6,
    "router_accuracy": 1.0,
    "faithfulness": 4.0,
    "correctness": 3.1666666666666665,
    "relevance": 3.6666666666666665
  },
  "by_language": {
    "ar": {
      "n": 27,
      "faithfulness": 4.0,
      "correctness": 3.814814814814815,
      "relevance": 4.0
    },
    "en": {
      "n": 30,
      "faithfulness": 3.533333333333333,
      "correctness": 3.3666666666666667,
      "relevance": 3.8
    },
    "fr": {
      "n": 4,
      "fai

In [7]:
report_groups = {
    "Overall": df,
    "Text": df[df["expected_mode"] == "text"],
    "Visual": df[df["expected_mode"] == "visual"],
    "Cross-language": df[df["cross_language"] == True],
}

table_rows = []

for name, group in report_groups.items():
    table_rows.append({
        "Evaluation group": name,
        "Cases": len(group),
        "Routing accuracy": group["router_ok"].mean(),
        "Faithfulness (/4)": group["faithfulness"].mean(),
        "Correctness (/4)": group["correctness"].mean(),
        "Relevance (/4)": group["relevance"].mean(),
    })

report_table = pd.DataFrame(table_rows)


display(report_table)

,Evaluation group,Cases,Routing accuracy,Faithfulness (/4),Correctness (/4),Relevance (/4)
0,Overall,61,0.918033,3.770492,3.573770,3.901639
1,Text,51,0.960784,3.941176,3.627451,3.882353
2,Visual,10,0.700000,2.900000,3.300000,4.000000
3,Cross-language,6,1.000000,4.000000,3.166667,3.666667
